In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS
# ==============================================================================
url_github = "https://raw.githubusercontent.com/humbertoeliaslopes/AD_negocios/09c51016706f6a932880621f9f519aa9ce1caf97/2011Movies.xlsx"

try:
    df = pd.read_excel(url_github)
except Exception:
    df = pd.read_excel("2011Movies.xlsx")

col_abertura = "Opening Gross Sales ($millions)"
col_total = "Total Gross Sales ($millions)"
col_salas = "Number of Theaters"
col_semanas = "Weeks in Release"

total_obs = len(df)


# ==============================================================================
# 2. FUNÇÃO: TABELA DE DISTRIBUIÇÃO DE FREQUÊNCIAS (ABNT + STORYTELLING)
# ==============================================================================
def tabela_frequencia(coluna, bins, titulo_tabela, subtitulo_tabela):
    rotulos = [f"[{bins[i]}, {bins[i+1]})" for i in range(len(bins) - 2)]
    rotulos.append(f"[{bins[-2]}, {bins[-1]}]")
    pontos_medios = [(bins[i] + bins[i + 1]) / 2 for i in range(len(bins) - 1)]

    classe = pd.cut(df[coluna], bins=bins, right=False, include_lowest=True)
    freq_abs = pd.Series(classe.value_counts(sort=False).values, index=rotulos)
    freq_rel = freq_abs / total_obs
    freq_perc = freq_rel * 100
    freq_acum = freq_abs.cumsum()
    freq_perc_acum = freq_perc.cumsum()

    tabela = pd.DataFrame({
        "Frequência Absoluta": freq_abs,
        "Frequência Relativa": freq_rel,
        "Frequência Percentual (%)": freq_perc,
        "Frequência Acumulada": freq_acum,
        "Frequência Percentual Acumulada (%)": freq_perc_acum,
    })
    tabela.index.name = "Intervalo de Classe"

    tabela.loc["Total"] = [
        freq_abs.sum(), freq_rel.sum(), freq_perc.sum(), np.nan, np.nan
    ]

    legenda_html = (
        f"<div style='text-align: left; margin-bottom: 8px; font-family: Arial, sans-serif;'>"
        f"<strong style='font-size: 11pt; color: black;'>{titulo_tabela}</strong><br>"
        f"<span style='font-size: 9.5pt; font-style: italic; color: #555555;'>{subtitulo_tabela}</span>"
        f"</div>"
    )

    tabela_estilizada = (
        tabela.style
        .format({
            "Frequência Absoluta": lambda x: f"{x:,.0f}" if pd.notnull(x) else "—",
            "Frequência Relativa": lambda x: f"{x:.2f}".replace(".", ",") if pd.notnull(x) else "—",
            "Frequência Percentual (%)": lambda x: f"{x:.2f}%".replace(".", ",") if pd.notnull(x) else "—",
            "Frequência Acumulada": lambda x: f"{x:,.0f}" if pd.notnull(x) else "—",
            "Frequência Percentual Acumulada (%)": lambda x: f"{x:.2f}%".replace(".", ",") if pd.notnull(x) else "—",
        })
        .set_table_styles([
            {"selector": "", "props": [("background-color", "white")]},
            {"selector": "caption", "props": [("caption-side", "top"), ("text-align", "left"), ("padding-bottom", "6px")]},
            {"selector": "thead", "props": [("border-top", "2px solid black"), ("border-bottom", "1px solid black"), ("background-color", "white")]},
            {"selector": "th", "props": [("text-align", "center"), ("font-weight", "bold"), ("background-color", "white"), ("color", "black"), ("padding", "6px 10px")]},
            {"selector": "tbody tr:last-child", "props": [("border-top", "1px solid black"), ("border-bottom", "2px solid black"), ("font-weight", "bold"), ("background-color", "white"), ("color", "black")]},
            {"selector": "td", "props": [("border", "none"), ("text-align", "right"), ("padding", "6px 12px"), ("background-color", "white"), ("color", "black")]},
            {"selector": "th.row_heading", "props": [("text-align", "left"), ("font-weight", "normal"), ("border", "none"), ("background-color", "white"), ("color", "black")]},
        ])
        .set_caption(legenda_html)
    )

    return tabela_estilizada, freq_abs, pontos_medios


# ==============================================================================
# 3. FUNÇÃO: HISTOGRAMA COM STORYTELLING (KNAFLIC)
# ==============================================================================
def histograma(freq_abs, pontos_medios, amplitude, titulo, subtitulo, xlabel, sufixo_x, nome_arquivo):
    pontos_medios = np.array(pontos_medios)
    frequencias = freq_abs.values[:len(pontos_medios)]

    ordenadas = sorted(frequencias, reverse=True)
    maior, segunda = ordenadas[0], ordenadas[1] if len(ordenadas) > 1 else -1

    cores = []
    for f in frequencias:
        if f == maior:
            cores.append("#1A365D")
        elif f == segunda and f > 0:
            cores.append("#2B6CB0")
        elif f > 0:
            cores.append("#718096")
        else:
            cores.append("#E2E8F0")

    fig, ax = plt.subplots(figsize=(10.5, 5.8), dpi=300)
    fig.patch.set_facecolor("#FFFFFF")
    ax.set_facecolor("#FFFFFF")

    barras = ax.bar(
        pontos_medios, frequencias, width=amplitude * 0.96,
        color=cores, edgecolor="#FFFFFF", linewidth=1.2, align="center"
    )

    for barra, freq in zip(barras, frequencias):
        if freq > 0:
            pct_val = (freq / total_obs) * 100
            ax.text(
                barra.get_x() + barra.get_width() / 2, barra.get_height() + max(frequencias) * 0.015,
                f"{int(freq)} ({pct_val:.0f}%)", ha="center", va="bottom",
                fontsize=9, fontweight="bold", color="#2D3748"
            )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_color("#718096")
    ax.spines["bottom"].set_linewidth(1.1)
    ax.yaxis.set_visible(False)

    limites_x = list(np.arange(pontos_medios[0] - amplitude / 2, pontos_medios[-1] + amplitude, amplitude))
    ax.set_xticks(limites_x)
    ax.set_xticklabels([f"{int(v):,}{sufixo_x}".replace(",", ".") for v in limites_x], fontsize=8.5, color="#4A5568")
    ax.tick_params(axis="x", length=4, color="#718096")
    ax.set_xlabel(xlabel, fontsize=10, color="#4A5568", labelpad=10)
    ax.set_ylim(0, max(frequencias) * 1.18)

    fig.text(0.09, 0.96, titulo, fontsize=12, fontweight="bold", color="#1A202C")
    fig.text(0.09, 0.915, subtitulo, fontsize=9, color="#718096")

    plt.subplots_adjust(top=0.85)
    plt.savefig(nome_arquivo, dpi=300, bbox_inches="tight")
    plt.show()


# ==============================================================================
# 4. FUNÇÃO: DIAGRAMA DE DISPERSÃO COM LINHA DE TENDÊNCIA (KNAFLIC)
# ==============================================================================
def dispersao(x, y, destaque_mascara, rotulo_destaque, titulo, subtitulo, xlabel, ylabel, nome_arquivo):
    inclinacao, intercepto = np.polyfit(x, y, 1)
    correlacao = np.corrcoef(x, y)[0, 1]
    r_quadrado = correlacao ** 2

    fig, ax = plt.subplots(figsize=(10, 5.8), dpi=300)
    fig.patch.set_facecolor("#FFFFFF")
    ax.set_facecolor("#FFFFFF")

    ax.scatter(x[~destaque_mascara], y[~destaque_mascara], color="#A0AEC0", alpha=0.7, s=45,
               edgecolors="none", label="Demais filmes")
    ax.scatter(x[destaque_mascara], y[destaque_mascara], color="#1A365D", alpha=0.9, s=65,
               edgecolors="#FFFFFF", linewidth=0.8, label=rotulo_destaque)

    x_reta = np.linspace(x.min(), x.max(), 100)
    y_reta = intercepto + inclinacao * x_reta
    ax.plot(x_reta, y_reta, color="#d95f02", linewidth=2.2, zorder=4)

    fig.text(0.09, 0.96, titulo, fontsize=12, fontweight="bold", color="#1A202C")
    fig.text(0.09, 0.915, subtitulo, fontsize=9, color="#718096")

    ax.annotate(
        f"Total = {intercepto:.1f} + {inclinacao:.2f} × ({xlabel.split(' (')[0]})\nR² = {r_quadrado:.2f}  |  r = {correlacao:.2f}",
        xy=(x.quantile(0.75), intercepto + inclinacao * x.quantile(0.75)),
        xytext=(x.quantile(0.55), y.max() * 0.85),
        arrowprops=dict(arrowstyle="->", color="#333333", lw=1),
        fontsize=8.5, color="#222222",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#f9f9f9", edgecolor="#cccccc", alpha=0.95),
    )

    ax.set_xlabel(xlabel, fontsize=10, color="#333333", labelpad=8)
    ax.set_ylabel(ylabel, fontsize=10, color="#333333", labelpad=8)
    ax.grid(axis="y", linestyle=":", color="#e0e0e0", alpha=0.8, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#888888")
    ax.spines["bottom"].set_color("#888888")
    ax.legend(frameon=False, loc="lower right", fontsize=8.5)

    plt.subplots_adjust(top=0.85)
    plt.savefig(nome_arquivo, dpi=300, bbox_inches="tight")
    plt.show()
    return inclinacao, intercepto, correlacao


# ==============================================================================
# 5. ITEM 1: TABELAS E HISTOGRAMAS DAS 4 VARIÁVEIS
# ==============================================================================
tab_abertura, freq_abertura, pm_abertura = tabela_frequencia(
    col_abertura, list(range(0, 200, 25)),
    "Tabela 1: Distribuição de Frequências das Vendas de Abertura",
    "Vendas brutas no fim de semana de estreia, em US$ milhões (n = 100)"
)
display(tab_abertura)
histograma(
    freq_abertura, pm_abertura, 25,
    "Estreias Concentradas: 63% dos Filmes Abrem com Menos de US$ 25 Milhões",
    "Distribuição das vendas brutas no fim de semana de estreia (n = 100 filmes, 2011)",
    "Vendas de Abertura (US$ milhões)", "M", "hist_abertura.png"
)

tab_total, freq_total, pm_total = tabela_frequencia(
    col_total, list(range(0, 450, 50)),
    "Tabela 2: Distribuição de Frequências das Vendas Brutas Totais",
    "Vendas brutas totais, em US$ milhões (n = 100)"
)
display(tab_total)
histograma(
    freq_total, pm_total, 50,
    "Bilheteria Concentrada: 70% dos Filmes Faturam Até US$ 100 Milhões no Total",
    "Distribuição das vendas brutas totais (n = 100 filmes, 2011)",
    "Vendas Brutas Totais (US$ milhões)", "M", "hist_total.png"
)

tab_salas, freq_salas, pm_salas = tabela_frequencia(
    col_salas, list(range(1000, 5000, 500)),
    "Tabela 3: Distribuição de Frequências do Número de Salas de Cinema",
    "Número de salas em que o filme foi exibido (n = 100)"
)
display(tab_salas)
histograma(
    freq_salas, pm_salas, 500,
    "Lançamento Amplo: 82% dos Filmes Estreiam Entre 2.500 e 4.000 Salas",
    "Distribuição do número de salas de cinema (n = 100 filmes, 2011)",
    "Número de Salas de Cinema", "", "hist_salas.png"
)

tab_semanas, freq_semanas, pm_semanas = tabela_frequencia(
    col_semanas, list(range(5, 50, 5)),
    "Tabela 4: Distribuição de Frequências das Semanas em Lançamento",
    "Número de semanas em que o filme esteve em cartaz (n = 100)"
)
display(tab_semanas)
histograma(
    freq_semanas, pm_semanas, 5,
    "Ciclo Curto: 78% dos Filmes Ficam em Cartaz Entre 10 e 20 Semanas",
    "Distribuição das semanas em lançamento (n = 100 filmes, 2011)",
    "Semanas em Lançamento", "sem", "hist_semanas.png"
)

# ==============================================================================
# 6. ITENS 2, 3 e 4: DIAGRAMAS DE DISPERSÃO
# ==============================================================================
blockbuster = df[col_total] > 200

inc1, int1, r1 = dispersao(
    df[col_abertura], df[col_total], blockbuster, "Blockbusters (Total > US$ 200M)",
    "Abertura Forte Prevê Bilheteria: Correlação de 0,89 com o Total Arrecadado",
    "Vendas de abertura x vendas brutas totais (n = 100 filmes, 2011)",
    "Vendas de Abertura (US$ milhões)", "Vendas Brutas Totais (US$ milhões)",
    "disp_abertura_total.png"
)

inc2, int2, r2 = dispersao(
    df[col_salas], df[col_total], blockbuster, "Blockbusters (Total > US$ 200M)",
    "Mais Salas, Mais Bilheteria: Correlação Moderada de 0,64",
    "Número de salas x vendas brutas totais (n = 100 filmes, 2011)",
    "Número de Salas de Cinema", "Vendas Brutas Totais (US$ milhões)",
    "disp_salas_total.png"
)

inc3, int3, r3 = dispersao(
    df[col_semanas], df[col_total], blockbuster, "Blockbusters (Total > US$ 200M)",
    "Tempo em Cartaz Importa Pouco: Correlação Fraca de 0,33",
    "Semanas em lançamento x vendas brutas totais (n = 100 filmes, 2011)",
    "Semanas em Lançamento", "Vendas Brutas Totais (US$ milhões)",
    "disp_semanas_total.png"
)

print(f"Correlações -> Abertura: {r1:.4f} | Salas: {r2:.4f} | Semanas: {r3:.4f}")
print(f"Blockbusters (Total > US$200M): {blockbuster.sum()} filmes ({blockbuster.sum()}%)")
